In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)

In [ ]:
import re
import glob
import os
import subprocess
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from openbabel import openbabel, pybel
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AddHs
from rdkit.Chem.rdmolfiles import MolToSmiles
from openbabel import openbabel, pybel
from rdkit.Chem import RemoveHs
import tempfile
from scipy.signal import argrelextrema
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
import xdrlib
from MDAnalysis.lib.util import NamedStream
import random
import time
import imageio
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures import ThreadPoolExecutor
import threading
import MDAnalysis as mda
import xdrlib
from MDAnalysis.lib.util import NamedStream
import signal
from scipy.signal import medfilt
import shutil
import zipfile
from numpy.fft import fftn, fftfreq
import matplotlib.pyplot as plt

In [ ]:
import resource

os.environ["OMP_STACKSIZE"] = "256M"
os.environ["KMP_STACKSIZE"] = "256M"


resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY))

In [ ]:



df = pd.read_excel('System.xlsx')



center_name = df['center atom'].iloc[0]
Temperature = df['temperature (K)'].iloc[0]
final_time = df['time (ns)'].iloc[0]


print(f"center atom: {center_name}")
print(f"Temperature: {Temperature} K")
print(f"Time: {final_time} ns")

In [ ]:
def return_molecule_polymer_list(df):
    
    
    molecule_name = []
    polymer_name = []
    
    
    for index, row in df.iterrows():
        
        if row['is polymer'] == False:
            
            name = str(row['Name'])
            molecule_name.append(name)
            
    
    for index, row in df.iterrows():
        
        if row['is polymer']:
            
            name = str(row['Name'])
            polymer_name.append(name)
    
    
    print("Molecule names:", molecule_name)
    print("Polymer names:", polymer_name)
    return molecule_name, polymer_name

In [ ]:
molecule_name, polymer_name = return_molecule_polymer_list(df)

In [ ]:
df

In [ ]:

def get_atom_count_from_mol2(mol2_path):
    
    
    atom_count_regex = re.compile(r'^\s*\d+\s+\d+')
    with open(mol2_path, 'r') as file:
        for line in file:
            
            if atom_count_regex.match(line):
                
                atom_count = int(line.split()[0])
                return atom_count
    return 0  

In [ ]:


def generate_index_commands(excel_path):
    
    
    df = pd.read_excel(excel_path)
    
    
    index_commands = {}
    
    
    current_atom_count = 0
    
    
    for _, row in df.iterrows():
        
        name = row['Name']
        number = row['Number']
        
        
        mol2_path = f"{name}.mol2"
        
        
        n_atoms = get_atom_count_from_mol2(mol2_path)
        
        
        total_atoms = n_atoms * number
        
        
        start_index = current_atom_count + 1
        end_index = current_atom_count + total_atoms
        index_command = f"a {start_index} - {end_index}"
        
        
        current_atom_count += total_atoms
        
        
        index_commands[name] = index_command
    
    return index_commands

In [ ]:

index_commands = generate_index_commands("System.xlsx")
print(index_commands)

In [ ]:
center_name

In [ ]:

def read_charges_and_atoms_from_itp(df, itp_file):
    
    
    serial_to_name = dict(zip(df['Name'], df['Serial Number'].fillna(0).astype(int)))
    
    
    charges_dict = {}
    
    
    name = os.path.splitext(itp_file)[0]  
    if name in serial_to_name:  
        with open(itp_file, 'r') as file:
            lines = file.readlines()
            for line in lines:
                
                match = re.match(r'(\s*\d+\s*)(\w+)\s+(\d+)\s+(\w+)\s+(\w+_\d+)\s+(\d+)\s+([-+]?\d*\.\d+)\s+([-+]?\d*\.\d+)', line)
                if match:
                    atom_col = match.group(5) 
                    charge_col = match.group(7) 
                    charge_col = float(charge_col) 
                    charges_dict[atom_col] = charge_col 

    return charges_dict

In [ ]:

def generate_index_commands_for_rdf_withatom(df):
    
    
    center_atom = df['center atom'].iloc[0]  
    
    
    row = df[df['Name'] == center_atom]
    
    
    if not row.empty:
        row = row.iloc[0]
        
        net_charge = row['Net Charge']
    
        
        if net_charge >= 0: 
            print("Public message removed for release.")
            
            
            itp_files = sorted([                      
                f for f in glob.glob('*.itp')         
                if os.path.isfile(f)                  
                and not os.path.basename(f).startswith('without_charge')  
            ])

            
            min_charge_atom_dict = {}

            
            min_charge_atom_index_command_dict = {}
            
            for itp_file in itp_files:
                
                base_filename = os.path.splitext(itp_file)[0]

                print(base_filename)

                
                charges_dict = read_charges_and_atoms_from_itp(df, itp_file)
                
                print(charges_dict)

                
                min_charge = float('inf')  
                min_charge_atom = None  
                
                
                for atom, charge in charges_dict.items():
                    if charge < min_charge:
                        min_charge = charge
                        min_charge_atom = atom
                        min_charge_atom_index_command = f"a {atom}"
                        
                min_charge_atom_dict[base_filename] = min_charge_atom
                min_charge_atom_index_command_dict[base_filename] = min_charge_atom_index_command
                
            return  min_charge_atom_index_command_dict, min_charge_atom_dict

        
        elif net_charge < 0: 
            print("Public message removed for release.")
            
            
            itp_files = sorted([                      
                f for f in glob.glob('*.itp')         
                if os.path.isfile(f)                  
                and not os.path.basename(f).startswith('without_charge')  
            ])

            
            max_charge_atom_dict = {}

            
            max_charge_atom_index_command_dict = {}
            
            for itp_file in itp_files:
                
                base_filename = os.path.splitext(itp_file)[0]
                
                charges_dict = read_charges_and_atoms_from_itp(df, itp_file)
            
                
                max_charge = -float('inf')  
                max_charge_atom = None  
                
                
                for atom, charge in charges_dict.items():
                    if charge > max_charge:
                        max_charge = charge
                        max_charge_atom = atom
                        max_charge_atom_index_command = f"a {atom}"
                        
                max_charge_atom_dict[base_filename] = max_charge_atom
                max_charge_atom_index_command_dict[base_filename] = max_charge_atom_index_command
            
            return  max_charge_atom_index_command_dict, max_charge_atom_dict
            
    else:
        print("Public message removed for release.")

In [ ]:

charge_atom_index_command_dict, charge_atom_dict = generate_index_commands_for_rdf_withatom(df)
print(f"charge_atom_dict: {charge_atom_dict}") 
print(f"charge_atom_index_command_dict: {charge_atom_index_command_dict}") 

In [ ]:
def _parse_ndx_groups(ndx_file):
    
    groups = []
    current_group = None

    with open(ndx_file, 'r', encoding='utf-8', errors='ignore') as fh:
        for raw_line in fh:
            line = raw_line.strip()
            if not line:
                continue

            match = re.match(r'^\[\s*(.*?)\s*\]$', line)
            if match:
                current_group = {'name': match.group(1), 'atoms': []}
                groups.append(current_group)
                continue

            if current_group is not None:
                current_group['atoms'].extend(line.split())

    return groups


def _format_ndx_group(name, atoms, atoms_per_line=15):
    
    output_lines = [f'[ {name} ]\n']
    for start in range(0, len(atoms), atoms_per_line):
        chunk = atoms[start:start + atoms_per_line]
        output_lines.append(''.join(f'{atom:>6}' for atom in chunk) + '\n')
    return output_lines


def clean_component_index_file(ndx_file, component_names, keep_system=True):
    
    groups = _parse_ndx_groups(ndx_file)
    if not groups:
        raise ValueError("Public message removed for release.")

    groups_by_name = {}
    for group in groups:
        groups_by_name.setdefault(group['name'], []).append(group)

    clean_groups = []
    name_to_index = {}

    if keep_system:
        system_candidates = groups_by_name.get('System', [])
        if not system_candidates:
            raise ValueError("Public message removed for release.")
        clean_groups.append({'name': 'System', 'atoms': system_candidates[0]['atoms']})
        name_to_index['System'] = 0

    component_set = set(str(name) for name in component_names)
    for component_name in [str(name) for name in component_names]:
        candidates = groups_by_name.get(component_name, [])
        if not candidates:
            available = ', '.join(group['name'] for group in groups)
            raise ValueError(
                "Public message removed for release."
                "Public message removed for release."
            )

        
        selected_group = candidates[-1]
        if not selected_group['atoms']:
            raise ValueError("Public message removed for release.")

        clean_groups.append({'name': component_name, 'atoms': selected_group['atoms']})
        name_to_index[component_name] = len(clean_groups) - 1

    with open(ndx_file, 'w', encoding='utf-8', newline='\n') as fh:
        for group_index, group in enumerate(clean_groups):
            if group_index > 0:
                fh.write('\n')
            fh.writelines(_format_ndx_group(group['name'], group['atoms']))

    duplicate_names = sorted(
        name for name, candidates in groups_by_name.items()
        if len(candidates) > 1 and name in component_set
    )
    if duplicate_names:
        print("Public message removed for release.")
    print("Public message removed for release.")
    return name_to_index


def create_index_file(index_commands, tpr_file='prod_NVT.tpr', ndx_file='index.ndx', excel_file='System.xlsx'):
    
    cmd = ['gmx', 'make_ndx', '-f', tpr_file, '-o', ndx_file]
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    input_commands = '\n'.join(
        f'{selection}\nname {group_index} {name}'
        for group_index, (name, selection) in enumerate(index_commands.items(), start=1)
    )
    input_commands += '\nq\n'
    stdout, stderr = process.communicate(input=input_commands)
    if process.returncode != 0:
        raise RuntimeError("Public message removed for release.")

    component_names = pd.read_excel(excel_file)['Name'].astype(str).tolist()
    return clean_component_index_file(ndx_file, component_names, keep_system=True)


In [ ]:
def create_chargeatom_index_file(index_commands, tpr_file='prod_NVT.tpr', ndx_file='index_atomcharge.ndx', excel_file='System.xlsx'):
    
    cmd = ['gmx', 'make_ndx', '-f', tpr_file, '-o', ndx_file]
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    input_commands = '\n'.join(
        f'{selection}\nname {group_index} {name}'
        for group_index, (name, selection) in enumerate(index_commands.items(), start=1)
    )
    input_commands += '\nq\n'
    stdout, stderr = process.communicate(input=input_commands)
    if process.returncode != 0:
        raise RuntimeError("Public message removed for release.")

    component_names = pd.read_excel(excel_file)['Name'].astype(str).tolist()
    return clean_component_index_file(ndx_file, component_names, keep_system=True)


In [ ]:

name_to_index = create_index_file(index_commands)
print(name_to_index)


In [ ]:

name_to_index_atomcharge = create_chargeatom_index_file(charge_atom_index_command_dict)
print(name_to_index_atomcharge)


In [ ]:
def update_system_xlsx(name_to_index, excel_file='System.xlsx'):
    df = pd.read_excel(excel_file)
    df['Index Group Number'] = df['Name'].apply(lambda name: name_to_index.get(name, ''))
    df.to_excel('System.xlsx', index=False)

In [ ]:
def update_chargeatom_system_xlsx(name_to_index_atomcharge, excel_file='System.xlsx'):
    df = pd.read_excel(excel_file)
    df['Atom Charge Index Group Number'] = df['Name'].apply(lambda name: name_to_index_atomcharge.get(name, ''))
    df.to_excel('System.xlsx', index=False)

In [ ]:

update_system_xlsx(name_to_index)

update_chargeatom_system_xlsx(name_to_index_atomcharge)


In [ ]:

df = pd.read_excel('System.xlsx')
df

In [ ]:

df['Diffusion Coefficient (cm^2/s)'] = None
df['Error (cm^2/s)'] = None

In [ ]:

assert 'Name' in df.columns and 'Index Group Number' in df.columns, "Excel file must contain 'Name' and 'Index Group Number' columns."

In [ ]:

name_to_index = pd.Series(df['Index Group Number'].values, index=df['Name']).to_dict()

In [ ]:
name_to_index

In [ ]:

def extract_diffusion_coefficient(msd_xvg_file_path):
    
    
    with open(msd_xvg_file_path, 'r') as file:
        content = file.read()

    
    
    pattern = r'D\[\s*.*?\s*\]\s*=\s*([\d\.\+\-eE]+)\s*\(\+/-\s*([\d\.\+\-eE]+)\)'

    
    match = re.search(pattern, content)

    if match:
        
        diffusion_coefficient_str = match.group(1)
        error_str = match.group(2)

        
        diffusion_coefficient = float(diffusion_coefficient_str) * 1e-5
        error = float(error_str) * 1e-5

        
        return diffusion_coefficient, error
    else:
        
        return None, None

In [ ]:
import re
import subprocess
from typing import Tuple

FLOAT_RE = r'[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eEdD][+-]?\d+)?'

DIFFUSION_SCALED_RE = re.compile(
    rf'^\s*(?:@\s*s\d+\s+legend\s+")?'
    rf'D\s*\[\s*(?P<label>[^\]]+?)\s*\]\s*=\s*'
    rf'(?P<D>{FLOAT_RE})\s*'
    rf'\(\s*[^)]*?(?P<err>{FLOAT_RE})[^)]*\)\s*'
    rf'\(\s*(?P<scale>{FLOAT_RE})\s*cm\^2/s\s*\)"?',
    re.IGNORECASE,
)

DIFFUSION_NM_RE = re.compile(
    rf'^\s*(?:@\s*s\d+\s+legend\s+")?'
    rf'D\s*\[\s*(?P<label>[^\]]+?)\s*\]\s*=\s*'
    rf'(?P<D>{FLOAT_RE})\s*'
    rf'\(\s*[^)]*?(?P<err>{FLOAT_RE})[^)]*\)\s*'
    rf'\(\s*nm\^2/ps\s*\)"?',
    re.IGNORECASE,
)

def _extract_from_text_to_cm2s(text: str) -> Tuple[float, float]:
    
    last_D = None          
    last_err = None        
    last_is_scaled = False 
    last_scale = 1.0       

    
    for line in text.splitlines():
        
        m_scaled = DIFFUSION_SCALED_RE.search(line)
        if m_scaled:
            last_D = float(m_scaled.group('D'))           
            last_err = float(m_scaled.group('err'))       
            last_scale = float(m_scaled.group('scale'))   
            last_is_scaled = True                         
            continue                                      

        
        m_nm = DIFFUSION_NM_RE.search(line)
        if m_nm:
            last_D = float(m_nm.group('D'))               
            last_err = float(m_nm.group('err'))           
            last_is_scaled = False                        
            last_scale = 1.0                              

    
    if last_D is None or last_err is None:
        return 0.0, 0.0

    
    if last_is_scaled:
        
        D_cm2s = last_D * last_scale
        err_cm2s = last_err * last_scale
    else:
        
        D_cm2s = last_D * 1e-2
        err_cm2s = last_err * 1e-2

    return D_cm2s, err_cm2s

def parse_msd_xvg(name: str) -> Tuple[float, float]:
    

    path = f"{name}_msd.xvg"

    try:
        last_D = None          
        last_err = None        
        last_is_scaled = False 
        last_scale = 1.0       

        with open(path, 'r', encoding='utf-8', errors='ignore') as fh:
            for line in fh:
                
                if 'D[' not in line:
                    continue

                
                m1 = DIFFUSION_SCALED_RE.search(line)
                if m1:
                    last_D = float(m1.group('D'))             
                    last_err = float(m1.group('err'))         
                    last_scale = float(m1.group('scale'))     
                    last_is_scaled = True                     
                    continue                                  

                
                m2 = DIFFUSION_NM_RE.search(line)
                if m2:
                    last_D = float(m2.group('D'))             
                    last_err = float(m2.group('err'))         
                    last_is_scaled = False                    
                    last_scale = 1.0                          

        
        if last_D is None or last_err is None:
            print("last_D is None or last_err is None")
            return 0.0, 0.0

        
        if last_is_scaled:
            
            return last_D * last_scale, last_err * last_scale
        else:
            
            return last_D * 1e-2, last_err * 1e-2

    except FileNotFoundError:
        
        print("FileNotFoundError")
        return 0.0, 0.0

def run_gmx_msd(
    name: str,
    index_number: int,
    trr_file: str = 'prod_NVT.trr',
    tpr_file: str = 'prod_NVT.tpr',
    ndx_file: str = 'index.ndx',
    gmx_cmd: str = 'gmx',
    verbose: bool = False
) -> Tuple[float, float]:
    
    cmd = [
        gmx_cmd, 'msd',
        '-f', trr_file,
        '-s', tpr_file,
        '-n', ndx_file,
        '-o', f'{name}_msd.xvg'
    ]

    proc = subprocess.Popen(
        cmd,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    stdout_text, stderr_text = proc.communicate(input=f"{index_number}\n")

    if verbose:
        print(stdout_text)
        print(stderr_text)

    
    diffusion_coefficient, diff_error = _extract_from_text_to_cm2s(stdout_text)

    
    if diffusion_coefficient == 0.0 and diff_error == 0.0:
        diffusion_coefficient, diff_error = parse_msd_xvg(name)

    if verbose:
        print(f"{name} Diffusion Coefficient: {diffusion_coefficient} cm^2/s")
        print(f"{name} Error: {diff_error} cm^2/s")

    return diffusion_coefficient, diff_error

In [ ]:

def update_excel_diffusion_coefficient(name, diffusion_coefficient, error):
    
    lock = threading.Lock()
    with lock:
        
        df = pd.read_excel('System.xlsx', engine='openpyxl')
        

        
        df.loc[df['Name'] == name, 'Diffusion Coefficient (cm^2/s)'] = diffusion_coefficient
        df.loc[df['Name'] == name, 'Error (cm^2/s)'] = error
        
        df.to_excel('System.xlsx', index=False, engine='openpyxl')

In [ ]:
df = pd.read_excel('System.xlsx', engine='openpyxl')
df['Diffusion Coefficient (cm^2/s)'] = 0.0
df['Error (cm^2/s)'] = 0.0

with ThreadPoolExecutor() as executor:
    futures = []
    for name, index_number in name_to_index.items():
        
        future = executor.submit(run_gmx_msd, name, index_number)
        futures.append((name, future))

    
    for name, future in futures:
        result = future.result()
        if result is not None:
            diffusion_coefficient, error = result
            update_excel_diffusion_coefficient(name, diffusion_coefficient, error)
        else:
            print(f"Failed to get results for {name}")

In [ ]:

import pandas as pd  

def update_diffusion_coefficients(df: pd.DataFrame, name_to_index: dict) -> pd.DataFrame:
    
    
    if 'Diffusion Coefficient (cm^2/s)' not in df.columns:
        df['Diffusion Coefficient (cm^2/s)'] = pd.NA
    if 'Error (cm^2/s)' not in df.columns:
        df['Error (cm^2/s)'] = pd.NA

    
    for name, idx in name_to_index.items():
        
        msd_file = f"{name}_msd.xvg"
        try:
            
            diff_coef, error = extract_diffusion_coefficient(msd_file)
        except Exception as e:
            
            print("Public message removed for release.")
            continue

        
        df.loc[df['Name'] == name, 'Diffusion Coefficient (cm^2/s)'] = diff_coef
        df.loc[df['Name'] == name, 'Error (cm^2/s)'] = error

    return df  

In [ ]:
df = update_diffusion_coefficients(df, name_to_index)

df['Diffusion Coefficient (cm^2/s)'] = df['Diffusion Coefficient (cm^2/s)'].fillna(0.0)
df['Error (cm^2/s)'] = df['Error (cm^2/s)'].fillna(0.0)
df.to_excel('System.xlsx', index=False, engine='openpyxl')

df

In [ ]:

df['Viscosity (mPa·s)'] = 0.0

In [ ]:


def run_gmx_energy():
    cmd = ['gmx', 'energy', '-f', 'vis_NVT.edr', '-o', 'viscosity.xvg']
    
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    
    output, _ = process.communicate(input='\n')
    
    
    with open('gmx_energy_viscosity_output.txt', 'w') as f:
        f.write(output)
    
    
    match = re.search(r'(\d+)\s+1/Viscosity', output)
    if match:
        viscosity_number = match.group(1)
    else:
        print("Public message removed for release.")
        return None, None

    
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    output, _ = process.communicate(input=f'{viscosity_number}\n\n')

    
    match = re.search(r'1/Viscosity\s+(\d+\.\d+)', output)
    if match:
        viscosity_str = match.group(1)
        
        viscosity = (1 / float(viscosity_str)) * 1000
        return viscosity
    else:
        print("Public message removed for release.")
        return None

In [ ]:

viscosity = run_gmx_energy()


if viscosity is not None:
    df['Viscosity (mPa·s)'] = viscosity


df.to_excel('System.xlsx', index=False)

In [ ]:
df = pd.read_excel('System.xlsx')
df

In [ ]:

def read_temperature_from_mdp(mdp_file='prod_NVT.mdp'):
    with open(mdp_file, 'r') as file:
        contents = file.read()
    match = re.search(r'ref-t\s*=\s*(\d+\.\d+)', contents)
    if match:
        T = float(match.group(1))  
    else:
        raise ValueError("Temperature not found in the .mdp file.")
    return T

In [ ]:
print(read_temperature_from_mdp())

In [ ]:

with open('md_NVT.gro', 'r') as file:
    for line in file:
        pass  
box_sizes = line.split()  
L_nm = sum(float(size) for size in box_sizes) / 3  
L_cm = L_nm * 1e-7  

In [ ]:

T = read_temperature_from_mdp()


kb = 1.380649e-16


xi = 2.837297


df['Viscosity (g/cm·s)'] = df['Viscosity (mPa·s)'] * 1e-3


df['Corrected Diffusion Coefficient (cm^2/s)'] = df['Diffusion Coefficient (cm^2/s)'] + (xi * kb * T) / (6 * 3.141592653589793 * df['Viscosity (g/cm·s)'] * L_cm)
print((xi * kb * T) / (6 * 3.141592653589793 * df['Viscosity (g/cm·s)'] * L_cm))

df.to_excel('System.xlsx', index=False)

In [ ]:
df = pd.read_excel('System.xlsx')
df

In [ ]:

def calculate_rdf(name, excel_file='System.xlsx', trr_file='prod_NVT.trr', tpr_file='prod_NVT.tpr', ndx_file='index.ndx'):
    
    df = pd.read_excel(excel_file)

    
    assert 'Name' in df.columns and 'Index Group Number' in df.columns, "Excel file must contain 'Name' and 'Index Group Number' columns."

    
    try:
        ref_index = df[df['Name'] == name]['Index Group Number'].values[0]
    except IndexError:
        raise ValueError(f"Name '{name}' not found in the Excel file.")

    
    all_indices = df['Index Group Number'].tolist()

    
    cmd = ['gmx', 'rdf', '-f', trr_file, '-s', tpr_file, '-n', ndx_file, '-o', f'{name}_rdf.xvg', '-cn', f'{name}_rdf_cn.xvg']
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    
    process.stdin.write(f"{ref_index}\n")
    for index in all_indices:
        
        process.stdin.write(f"{index}\n")

        process.stdin.flush()

    
    process.stdin.close()
    
    
    
    rdf_file = f'{name}_rdf.xvg'
    rdf_cn_file = f'{name}_rdf_cn.xvg'
    rdf_and_cn_file = f'{name}_rdf+cn' 
    
    
    def wait_for_files(*filenames, timeout=None):
        
        start_time = time.time()
        while True:
            
            all_files_exist = all(os.path.isfile(filename) for filename in filenames)
            if all_files_exist:
                break  
            
            
            if timeout is not None and (time.time() - start_time) > timeout:
                raise TimeoutError(f"Timeout while waiting for files: {filenames}")
            
            
            time.sleep(1)
    
    
    wait_for_files(f'{name}_rdf.xvg', f'{name}_rdf_cn.xvg')
    
    
    print(f"Files for {name} are ready for further processing.")
    
    
    return rdf_file, rdf_cn_file, rdf_and_cn_file

In [ ]:

rdf_file, rdf_cn_file, rdf_and_cn_file = calculate_rdf(center_name)

In [ ]:

def calculate_chargeatom_rdf(name, excel_file='System.xlsx', trr_file='prod_NVT.trr', tpr_file='prod_NVT.tpr', ndx_file='index_atomcharge.ndx'):
    
    df = pd.read_excel(excel_file)

    
    assert 'Name' in df.columns and 'Atom Charge Index Group Number' in df.columns, "Excel file must contain 'Name' and 'Atom Charge Index Group Number' columns."

    
    try:
        ref_index = df[df['Name'] == name]['Atom Charge Index Group Number'].values[0]
    except IndexError:
        raise ValueError(f"Name '{name}' not found in the Excel file.")

    
    all_indices = df['Atom Charge Index Group Number'].tolist()

    
    cmd = ['gmx', 'rdf', '-f', trr_file, '-s', tpr_file, '-n', ndx_file, '-o', f'{name}_atomcharge_rdf.xvg', '-cn', f'{name}_atomcharge_rdf_cn.xvg']
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    
    process.stdin.write(f"{ref_index}\n")
    for index in all_indices:
        
        process.stdin.write(f"{index}\n")

        process.stdin.flush()

    
    process.stdin.close()
    
    
    
    rdf_atomcharge_file = f'{name}_atomcharge_rdf.xvg'
    rdf_atomcharge_cn_file = f'{name}_atomcharge_rdf_cn.xvg'
    rdf_atomcharge_and_cn_file = f'{name}_atomcharge_rdf+cn' 
    
    
    def wait_for_files(*filenames, timeout=None):
        
        start_time = time.time()
        while True:
            
            all_files_exist = all(os.path.isfile(filename) for filename in filenames)
            if all_files_exist:
                break  
            
            
            if timeout is not None and (time.time() - start_time) > timeout:
                raise TimeoutError(f"Timeout while waiting for files: {filenames}")
            
            
            time.sleep(1)
    
    
    wait_for_files(f'{name}_atomcharge_rdf.xvg', f'{name}_atomcharge_rdf_cn.xvg')
    
    
    print(f"Files for {name} are ready for further processing.")
    
    
    return rdf_atomcharge_file, rdf_atomcharge_cn_file, rdf_atomcharge_and_cn_file

In [ ]:

rdf_atomcharge_file, rdf_atomcharge_cn_file, rdf_atomcharge_and_cn_file = calculate_chargeatom_rdf(center_name)

In [ ]:

def rdf_name(name, excel_file='System.xlsx'):
    
    df = pd.read_excel(excel_file)

    
    assert 'Name' in df.columns and 'Index Group Number' in df.columns and 'Atom Charge Index Group Number' in df.columns, "Excel file must contain 'Name' and 'Index Group Number' columns."

    
    rdf_file = f'{name}_rdf.xvg'
    rdf_cn_file = f'{name}_rdf_cn.xvg'
    rdf_and_cn_file = f'{name}_rdf+cn' 
    rdf_atomcharge_file = f'{name}_atomcharge_rdf.xvg'
    rdf_atomcharge_cn_file = f'{name}_atomcharge_rdf_cn.xvg'
    rdf_atomcharge_and_cn_file = f'{name}_atomcharge_rdf+cn' 
    
    
    if not os.path.isfile(f'{name}_rdf.xvg') or not os.path.isfile(f'{name}_rdf_cn.xvg'):
        raise RuntimeError(f"RDF calculation failed for {name}.")

    
    if not os.path.isfile(f'{name}_atomcharge_rdf.xvg') or not os.path.isfile(f'{name}_atomcharge_rdf_cn.xvg'):
        raise RuntimeError(f"Atom Charge RDF calculation failed for {name}.")
    
    
    return rdf_file, rdf_cn_file, rdf_and_cn_file, rdf_atomcharge_file, rdf_atomcharge_cn_file, rdf_atomcharge_and_cn_file


In [ ]:
rdf_file, rdf_cn_file, rdf_and_cn_file, rdf_atomcharge_file, rdf_atomcharge_cn_file, rdf_atomcharge_and_cn_file = rdf_name(center_name)

In [ ]:

def read_rdf_xvg(filename):
    
    legend_pattern = re.compile(r'^\s*@\s*s(\d+)\s+legend\s+"(.*)"\s*$')
    legends = {}
    data = []

    with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
        for raw_line in f:
            line = raw_line.strip()

            
            if not line:
                continue

            
            legend_match = legend_pattern.match(line)
            if legend_match:
                series_index = int(legend_match.group(1))
                legends[series_index] = legend_match.group(2).strip()
                continue

            
            if line[0] in ('@', '#'):
                continue

            try:
                data.append([float(value) for value in line.split()])
            except ValueError as exc:
                raise ValueError("Public message removed for release.") from exc

    if not data:
        raise ValueError("Public message removed for release.")

    df = pd.DataFrame(data)

    
    expected_legend_count = df.shape[1] - 1
    ordered_legends = [legends[idx] for idx in sorted(legends)]

    if expected_legend_count <= 0:
        raise ValueError("Public message removed for release.")

    if len(ordered_legends) != expected_legend_count:
        raise ValueError(
            "Public message removed for release."
            "Public message removed for release."
        )

    df.columns = ['r (nm)'] + ordered_legends
    return df


In [ ]:

def read_xvg(filename):
    
    data = []

    with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
        for raw_line in f:
            line = raw_line.strip()

            if not line or line[0] in ('@', '#'):
                continue

            data.append([float(value) for value in line.split()])

    return pd.DataFrame(data)


In [ ]:

df_rdf = read_rdf_xvg(rdf_file)
df_rdf_cn = read_rdf_xvg(rdf_cn_file)
df_rdf_atomcharge = read_rdf_xvg(rdf_atomcharge_file)
df_rdf_cn_atomcharge = read_rdf_xvg(rdf_atomcharge_cn_file)


In [ ]:

system_names = pd.read_excel('System.xlsx', usecols=['Name'])['Name'].tolist()

def validate_rdf_component_names(df, expected_names, df_label):
    
    actual_names = list(df.columns[1:])
    missing_names = [name for name in expected_names if name not in actual_names]
    extra_names = [name for name in actual_names if name not in expected_names]

    if missing_names or extra_names:
        raise ValueError(
            "Public message removed for release."
            "Public message removed for release."
        )

validate_rdf_component_names(df_rdf, system_names, 'df_rdf')
validate_rdf_component_names(df_rdf_cn, system_names, 'df_rdf_cn')
validate_rdf_component_names(df_rdf_atomcharge, system_names, 'df_rdf_atomcharge')
validate_rdf_component_names(df_rdf_cn_atomcharge, system_names, 'df_rdf_cn_atomcharge')


In [ ]:
df_rdf

In [ ]:
df_rdf_cn

In [ ]:

def equalize_dataframes(df1, df2):
    
    num_rows_df1 = df1.shape[0]
    num_rows_df2 = df2.shape[0]

    if num_rows_df1 != num_rows_df2:
        if num_rows_df1 > num_rows_df2:
            df1 = df1.iloc[:num_rows_df2]
        else:
            df2 = df2.iloc[:num_rows_df1]

    return df1, df2

In [ ]:

df_rdf_1, df_rdf_cn_1 = equalize_dataframes(df_rdf, df_rdf_cn)


x_data = df_rdf_1.iloc[:, 0]


fig, ax1 = plt.subplots(figsize=(3.5,3))

dpi = 300  


for column in df_rdf_1.columns[1:]:
    ax1.plot(x_data, df_rdf_1[column], label=column)


ax1.set_xlabel(df_rdf_1.columns[0])
ax1.set_ylabel('g(r)')
ax1.set_xlim(0, 2)  


ax2 = ax1.twinx()


for column in df_rdf_cn_1.columns[1:]:
    ax2.plot(x_data, df_rdf_cn_1[column], label=column, linestyle='--')


ax2.set_ylabel('coordination number')
ax2.set_ylim(0, 10)  


ax1.legend(loc='upper right')
ax2.legend(loc='upper left')

plt.tight_layout()  


fig.savefig(f"{rdf_and_cn_file}.png", format='png', dpi=dpi, bbox_inches='tight')


plt.show()

In [ ]:
df_rdf_atomcharge

In [ ]:
df_rdf_cn_atomcharge

In [ ]:

def update_column_headers(df, mapping_dict):
    new_columns = ['r (nm)']  
    for col in df.columns[1:]:  
        if col in mapping_dict:  
            new_col = f"{col}-{mapping_dict[col]}"  
            new_columns.append(new_col)  
        else:
            new_columns.append(col)  
    df.columns = new_columns  


update_column_headers(df_rdf_atomcharge, charge_atom_dict)
update_column_headers(df_rdf_cn_atomcharge, charge_atom_dict)

In [ ]:

df_rdf_atomcharge_1, df_rdf_cn_atomcharge_1 = equalize_dataframes(df_rdf_atomcharge, df_rdf_cn_atomcharge)


x_data = df_rdf_atomcharge_1.iloc[:, 0]


fig, ax1 = plt.subplots(figsize=(3.5,3))

dpi = 300  


for column in df_rdf_atomcharge_1.columns[1:]:
    ax1.plot(x_data, df_rdf_atomcharge_1[column], label=column)


ax1.set_xlabel(df_rdf_atomcharge_1.columns[0])
ax1.set_ylabel('g(r)')
ax1.set_xlim(0, 2)  


ax2 = ax1.twinx()


for column in df_rdf_cn_atomcharge_1.columns[1:]:
    ax2.plot(x_data, df_rdf_cn_atomcharge_1[column], label=column, linestyle='--')


ax2.set_ylabel('coordination number')
ax2.set_ylim(0, 10)  


ax1.legend(loc='upper right')
ax2.legend(loc='upper left')

plt.tight_layout()  


fig.savefig(f"{rdf_atomcharge_and_cn_file}.png", format='png', dpi=dpi, bbox_inches='tight')


plt.show()

In [ ]:

def run_gmx_density(trr_file='prod_NVT.trr', tpr_file='prod_NVT.tpr', ndx_file='index.ndx'):
    cmd = ['gmx', 'density', '-sl', '1', '-f', trr_file, '-s', tpr_file, '-n', ndx_file, '-o', 'density.xvg']
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    
    output, error = process.communicate(input='0\n')
    
    if error:
        print("Error while calculating viscosity:")
        print(error)
        return None

In [ ]:
run_gmx_density()

In [ ]:

def update_density_in_excel(density_file='density.xvg', excel_file='System.xlsx'):
    
    with open(density_file, 'r') as file:
        lines = file.readlines()
        last_line = lines[-1]
    
    
    density_value = last_line.split()[1]

    
    df = pd.read_excel(excel_file)

    
    df['Density (kg/m^3)'] = density_value

    
    df.to_excel('System.xlsx', index=False)

In [ ]:

update_density_in_excel()

In [ ]:
df = pd.read_excel('System.xlsx')
df

In [ ]:
def calculate_ion_corrected_migration_coefficient(excel_file='System.xlsx'):
    import warnings  

    
    df = pd.read_excel(excel_file)

    
    required_columns = ['Number', 'Corrected Diffusion Coefficient (cm^2/s)', 'Net Charge', 'Is Ion']
    for column in required_columns:
        assert column in df.columns, f"Excel file must contain '{column}' column."

    
    for col in ['Number', 'Corrected Diffusion Coefficient (cm^2/s)', 'Net Charge']:
        df[col] = pd.to_numeric(df[col], errors='coerce')  

    
    ions_df = df[df['Is Ion']].copy()

    
    total = (
        ions_df['Number'].fillna(0) *
        ions_df['Corrected Diffusion Coefficient (cm^2/s)'].fillna(0) *
        ions_df['Net Charge'].abs().fillna(0)
    ).sum()

    
    df['Corrected Migration Coefficient'] = 0.0

    
    if pd.isna(total) or total == 0:
        warnings.warn("Public message removed for release.")
    else:
        
        weights = (
            df['Number'].fillna(0) *
            df['Corrected Diffusion Coefficient (cm^2/s)'].fillna(0) *
            df['Net Charge'].abs().fillna(0)
        )
        
        ion_mask = df['Is Ion']  
        df.loc[ion_mask, 'Corrected Migration Coefficient'] = weights[ion_mask] / total

    
    df.to_excel('System.xlsx', index=False)

In [ ]:
import warnings  

def calculate_ion_migration_coefficient(excel_file='System.xlsx'):
    
    df = pd.read_excel(excel_file)

    
    required_columns = ['Number', 'Diffusion Coefficient (cm^2/s)', 'Net Charge', 'Is Ion']
    for column in required_columns:
        assert column in df.columns, f"Excel file must contain '{column}' column."

    
    ions_df = df[df['Is Ion']].copy()

    
    
    total = (
        ions_df['Number'] *                                
        ions_df['Diffusion Coefficient (cm^2/s)'] *        
        ions_df['Net Charge'].abs()                        
    ).sum()                                                

    
    if pd.isna(total) or total == 0:                       
        warnings.warn(                                     
            "Public message removed for release."
        )
        df['Migration Coefficient'] = 0.0                  
    else:                                                  
        df['Migration Coefficient'] = df.apply(
            lambda row: (
                row['Number'] * row['Diffusion Coefficient (cm^2/s)'] * abs(row['Net Charge']) / total
            ) if row['Is Ion'] else 0,
            axis=1
        )

    
    df['Migration Coefficient'] = df.apply(
        lambda row: (row['Number'] * row['Diffusion Coefficient (cm^2/s)'] * abs(row['Net Charge']) / total) if row['Is Ion'] else 0,
        axis=1
    )

    
    df.to_excel('System.xlsx', index=False)

In [ ]:
df = pd.read_excel('System.xlsx')
df

In [ ]:

calculate_ion_migration_coefficient()
calculate_ion_corrected_migration_coefficient()

In [ ]:
df = pd.read_excel('System.xlsx')
df

In [ ]:

def read_box_volume_from_gro(gro_file='prod_NVT.gro'):
    with open(gro_file, 'r') as file:
        for line in file:
            pass  
        box_dims = line.split()  
    
    V = float(box_dims[0]) * float(box_dims[1]) * float(box_dims[2]) * 1e-21
    return V

In [ ]:
print(read_box_volume_from_gro())

In [ ]:


E_CHARGE = 1.602176634e-19           
K_B      = 1.380649e-23              

def calculate_ionic_conductivity(excel_file: str = "System.xlsx") -> float:
    
    
    df = pd.read_excel(excel_file)

    
    V_cm3 = read_box_volume_from_gro()        
    V_m3  = V_cm3 * 1e-6                      
    T     = read_temperature_from_mdp()       

    
    required = ['Is Ion', 'Number', 'Net Charge',
                'Corrected Diffusion Coefficient (cm^2/s)']
    for col in required:
        assert col in df.columns, "Public message removed for release."

    
    ions = df[df['Is Ion'] == True].copy()

    
    ions['Di_m2s'] = ions['Diffusion Coefficient (cm^2/s)'] * 1e-4

    
    ions['ni_m3'] = ions['Number'] / V_m3

    
    summation = (ions['Net Charge']**2 * ions['Di_m2s'] * ions['ni_m3']).sum()

    
    sigma_Sm = (E_CHARGE**2 / (K_B * T)) * summation

    
    sigma_mScm = sigma_Sm * 10               

    print(f"ionic conductivity (Nernst–Einstein): {sigma_mScm:.3f} mS/cm")
    return sigma_mScm

In [ ]:

import pandas as pd


E_CHARGE = 1.602176634e-19           
K_B      = 1.380649e-23              

def calculate_corrected_ionic_conductivity(excel_file: str = "System.xlsx") -> float:
    
    
    df = pd.read_excel(excel_file)

    
    V_cm3 = read_box_volume_from_gro()        
    V_m3  = V_cm3 * 1e-6                      
    T     = read_temperature_from_mdp()       

    
    required = ['Is Ion', 'Number', 'Net Charge',
                'Corrected Diffusion Coefficient (cm^2/s)']
    for col in required:
        assert col in df.columns, "Public message removed for release."

    
    ions = df[df['Is Ion'] == True].copy()

    
    ions['Di_m2s'] = ions['Corrected Diffusion Coefficient (cm^2/s)'] * 1e-4

    
    ions['ni_m3'] = ions['Number'] / V_m3

    
    summation = (ions['Net Charge']**2 * ions['Di_m2s'] * ions['ni_m3']).sum()

    
    sigma_Sm = (E_CHARGE**2 / (K_B * T)) * summation

    
    sigma_mScm = sigma_Sm * 10               

    print(f"Corrected ionic conductivity (Nernst–Einstein): {sigma_mScm:.3f} mS/cm")
    return sigma_mScm

In [ ]:

ionic_conductivity = calculate_ionic_conductivity()
corrected_ionic_conductivity = calculate_corrected_ionic_conductivity()

In [ ]:
df['ionic_conductivity (mS/cm)'] = ionic_conductivity
df['corrected_ionic_conductivity (mS/cm)'] = corrected_ionic_conductivity

In [ ]:

df.to_excel('System.xlsx', index=False)

In [ ]:


def run_gmx_epsilon(trr_file='prod_NVT.trr', tpr_file='prod_NVT.tpr', temp=Temperature, delete_file=True):
    cmd = ['gmx', 'current', '-f', trr_file, '-s', tpr_file, '-temp', str(temp)]
    output_file = 'gmx_current_output.txt'
    
    with open(output_file, 'w') as file:
        process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=file, stderr=file, text=True)
        
        
        output, error = process.communicate(input='0\n')

    
    process.wait()
    
    
    with open(output_file, 'r') as file:
        output = file.read()
        match = re.search(r'Absolute values:\s*\n\s*epsilon=(\d+\.\d+)', output)
        if match:
            
            epsilon_value = match.group(1)
            print(f"Static dielectric constant: epsilon={epsilon_value}")
            return epsilon_value
        else:
            print("Could not find the static dielectric constant in the output.")
            return None
    
    
    if delete_file:
        os.remove(output_file)
    
    print("The 'Absolute values:' line could not be found in the output.")
    return None

In [ ]:

epsilon = run_gmx_epsilon()


if epsilon is not None:
    df['static dielectric constant'] = epsilon
    
    df.to_excel('System.xlsx', index=False)

In [ ]:
df

In [ ]:

T = read_temperature_from_mdp()  


k_b = 1.380649e-23

In [ ]:
df_rdf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter

def _choose_valid_savgol_window(n_points: int, preferred_window: int = 15, polyorder: int = 3) -> int:
    
    if n_points < 5:
        raise ValueError("Public message removed for release.")

    min_valid_window = polyorder + 2
    if min_valid_window % 2 == 0:
        min_valid_window += 1

    window = min(preferred_window, n_points if n_points % 2 == 1 else n_points - 1)
    if window < min_valid_window:
        window = min_valid_window
    if window > n_points:
        window = n_points if n_points % 2 == 1 else n_points - 1

    return max(window, min_valid_window)

def _select_first_shell_boundary(x_nm: np.ndarray,
                                 rdf_y: np.ndarray,
                                 cn_y: np.ndarray | None = None,
                                 smooth_window: int = 15,
                                 polyorder: int = 3,
                                 min_distance: int = 12) -> dict:
    
    x_nm = np.asarray(x_nm, dtype=float)
    y_raw = np.asarray(rdf_y, dtype=float)
    y_raw = np.nan_to_num(y_raw, nan=0.0, posinf=0.0, neginf=0.0)

    window = _choose_valid_savgol_window(len(y_raw), preferred_window=smooth_window, polyorder=polyorder)
    y_smooth = savgol_filter(y_raw, window_length=window, polyorder=polyorder, mode='interp')

    tail_start = max(int(len(y_smooth) * 0.8), 1)
    g_base = float(np.mean(y_smooth[tail_start:]))
    amplitude = max(float(np.max(y_smooth) - g_base), 1e-6)

    peak_prominence_threshold = max(0.10 * amplitude, 0.01)
    peaks, peak_props = find_peaks(
        y_smooth,
        prominence=peak_prominence_threshold,
        width=3,
    )
    peaks = peaks[x_nm[peaks] >= 0.05]
    if peaks.size == 0:
        raise ValueError("Public message removed for release.")

    first_peak = int(peaks[0])
    peak_height_value = float(y_smooth[first_peak])

    secondary_peak_prominence = max(0.10 * amplitude, 0.01)
    later_peaks, _ = find_peaks(y_smooth, prominence=secondary_peak_prominence, width=3)
    later_peaks = later_peaks[later_peaks > first_peak + min_distance]

    search_start = min(len(y_smooth) - 1, first_peak + min_distance)
    search_end = len(y_smooth) - 1

    if later_peaks.size > 0:
        search_end = int(later_peaks[0])
    else:
        baseline_return_threshold = g_base + max(0.10 * amplitude, 0.03)
        baseline_hits = np.where(y_smooth[search_start:] <= baseline_return_threshold)[0]
        if baseline_hits.size > 0:
            search_end = int(search_start + baseline_hits[0])

    if search_end <= search_start:
        search_end = min(len(y_smooth) - 1, first_peak + max(min_distance * 2, 20))
    if search_end <= search_start:
        raise ValueError("Public message removed for release.")

    valley_prominence_threshold = max(0.10 * amplitude, 0.01)
    valleys, _ = find_peaks(-y_smooth, prominence=valley_prominence_threshold, width=3)
    valley_candidates = [
        int(idx) for idx in valleys
        if search_start <= int(idx) <= search_end
    ]

    valley_depth_threshold = g_base + 0.25 * (peak_height_value - g_base)
    filtered_candidates = [idx for idx in valley_candidates if y_smooth[idx] <= valley_depth_threshold]
    candidate_pool = filtered_candidates if filtered_candidates else valley_candidates

    fallback_used = False
    if not candidate_pool:
        local_segment = y_smooth[search_start:search_end + 1]
        fallback_used = True
        if local_segment.size == 0:
            raise ValueError("Public message removed for release.")
        candidate_pool = [int(search_start + np.argmin(local_segment))]

    selected_valley = min(candidate_pool, key=lambda idx: y_smooth[idx])
    cn_validation = 'not_checked'

    if cn_y is not None:
        cn_array = np.asarray(cn_y, dtype=float)
        if cn_array.shape[0] == y_smooth.shape[0]:
            cn_grad = np.gradient(cn_array, x_nm)
            reference_slice = cn_grad[first_peak:selected_valley + 1]
            reference_grad = float(np.mean(np.abs(reference_slice))) if reference_slice.size > 0 else 0.0
            reference_grad = max(reference_grad, 1e-12)

            def _post_valley_grad(idx: int) -> float:
                tail = cn_grad[idx:min(idx + 5, len(cn_grad))]
                if tail.size == 0:
                    return reference_grad
                return float(np.mean(np.abs(tail)))

            current_post_grad = _post_valley_grad(selected_valley)
            if current_post_grad <= 0.35 * reference_grad:
                cn_validation = 'validated'
            else:
                validated_candidates = [
                    idx for idx in sorted(candidate_pool, key=lambda item: y_smooth[item])
                    if _post_valley_grad(idx) <= 0.35 * reference_grad
                ]
                if validated_candidates:
                    selected_valley = int(validated_candidates[0])
                    cn_validation = 'validated'
                else:
                    cn_validation = 'warning'

    return {
        'peak_index': int(first_peak),
        'valley_index': int(selected_valley),
        'peak_r_nm': float(x_nm[first_peak]),
        'valley_r_nm': float(x_nm[selected_valley]),
        'peak_height': float(y_smooth[first_peak]),
        'valley_height': float(y_smooth[selected_valley]),
        'cn_validation': cn_validation,
        'fallback_used': fallback_used,
        'search_start_index': int(search_start),
        'search_end_index': int(search_end),
    }

def extract_first_shell_radius(df_rdf: pd.DataFrame,
                               df_rdf_cn: pd.DataFrame | None = None,
                               smooth_window: int = 15,
                               polyorder: int = 3,
                               min_distance: int = 12) -> dict:
    
    x_nm = df_rdf['r (nm)'].to_numpy(dtype=float)
    results = {}

    for column in df_rdf.columns[1:]:
        cn_y = None
        if df_rdf_cn is not None and column in df_rdf_cn.columns:
            cn_y = df_rdf_cn[column].to_numpy(dtype=float)

        try:
            result = _select_first_shell_boundary(
                x_nm=x_nm,
                rdf_y=df_rdf[column].to_numpy(dtype=float),
                cn_y=cn_y,
                smooth_window=smooth_window,
                polyorder=polyorder,
                min_distance=min_distance,
            )
            results[column] = result
            print(
                f"{column}: peak = {result['peak_r_nm']:.4f} nm, "
                f"valley = {result['valley_r_nm']:.4f} nm, "
                f"cn_validation = {result['cn_validation']}, "
                f"fallback_used = {result['fallback_used']}"
            )
        except Exception as exc:
            print("Public message removed for release.")

    return results

extrema_indices = extract_first_shell_radius(
    df_rdf,
    df_rdf_cn=df_rdf_cn,
    smooth_window=15,
    polyorder=3,
    min_distance=12,
)


In [ ]:
print(extrema_indices)

In [ ]:

delta_G_a_values = {}

for column, shell_info in extrema_indices.items():
    
    
    if not isinstance(shell_info, dict):
        raise TypeError("Public message removed for release.")

    if 'peak_index' not in shell_info or 'valley_index' not in shell_info:
        raise KeyError("Public message removed for release.")

    max_idx = int(shell_info['peak_index'])
    min_idx = int(shell_info['valley_index'])

    g_max = float(df_rdf[column].iloc[max_idx])
    g_min = float(df_rdf[column].iloc[min_idx])

    
    if g_max <= 0 or g_min <= 0:
        print("Public message removed for release.")
        continue

    delta_G_a = k_b * T * np.log(g_max / g_min)
    
    delta_G_a_kJ_per_mol = delta_G_a * (6.02214076e23) / 1000
    delta_G_a_values[column] = float(delta_G_a_kJ_per_mol)



In [ ]:

df = pd.read_excel('System.xlsx')


df['Delta_G_a (kJ/mol)'] = df['Name'].map(delta_G_a_values)

df['Delta_G_a (kJ/mol)'] = df['Delta_G_a (kJ/mol)'].apply(lambda x: 0 if x < 0 else x)

df

In [ ]:

df.to_excel('System.xlsx', index=False)

In [ ]:
import subprocess
import concurrent.futures

def analyze_hbond(df, name_to_index, maxtask=4):
    

    
    
    
    
    
    
    
    
    result_dict = {}
    for idx, row in df.iterrows():
        name = row["Name"]
        smiles = row["SMILES"]

        
        if ("O" in smiles) or ("F" in smiles) or ("N" in smiles) or ("water" in name.lower()):
            
            index_number = name_to_index[name]  
            result_dict[index_number] = name

    
    if len(result_dict) == 0:
        print("Public message removed for release.")
        return

    
    
    
    
    
    index_numbers = list(result_dict.keys())
    tasks = []

    for i in range(len(index_numbers)):
        idx_i = index_numbers[i]
        name_i = result_dict[idx_i]

        
        tasks.append((idx_i, idx_i, name_i, name_i))

        
        for j in range(i + 1, len(index_numbers)):
            idx_j = index_numbers[j]
            name_j = result_dict[idx_j]
            tasks.append((idx_i, idx_j, name_i, name_j))
            
            

    
    def run_hbond_command(index1, index2, name1, name2):
        
        hbnum_file = f"hbnum_{name1}_{name2}.xvg"
        hblife_file = f"hblife_{name1}_{name2}.xvg"

        cmd = [
            "gmx", "hbond",
            "-f", "prod_NVT.trr",
            "-s", "prod_NVT.tpr",
            "-n", "index.ndx",
            "-num", hbnum_file,
            "-life", hblife_file
        ]

        process = subprocess.Popen(
            cmd,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        
        input_groups = f"{index1}\n{index2}\n"
        output, error = process.communicate(input=input_groups)

        
        if process.returncode != 0:
            print("Public message removed for release.")
        else:
            print("Public message removed for release.")
    
    
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=maxtask) as executor:
        future_to_task = {}
        for (i1, i2, n1, n2) in tasks:
            future = executor.submit(run_hbond_command, i1, i2, n1, n2)
            future_to_task[future] = (n1, n2)

        for future in concurrent.futures.as_completed(future_to_task):
            name_pair = future_to_task[future]
            try:
                future.result()
            except Exception as exc:
                print("Public message removed for release.")

    print("Public message removed for release.")

In [ ]:
analyze_hbond(df, name_to_index, maxtask=4)

In [ ]:

radius_dict_nm = {
    key: float(value['valley_r_nm'])
    for key, value in extrema_indices.items()
}

if not radius_dict_nm:
    raise RuntimeError("Public message removed for release.")

df_system = pd.read_excel('System.xlsx')
df_system['First Coordination Shell Radius (nm)'] = df_system['Name'].map(radius_dict_nm)
df_system.to_excel('System.xlsx', index=False)


header_for_min_extrema_str = min(radius_dict_nm, key=radius_dict_nm.get)
header_for_min_extrema = [header_for_min_extrema_str]
r_value_at_min_extrema = float(radius_dict_nm[header_for_min_extrema_str])

rdf_radius = r_value_at_min_extrema
print(
    f"first coordination sphere of {center_name}: {rdf_radius:.4f} nm, "
    f"find first coordination sphere in {center_name}_{header_for_min_extrema_str}"
)


rdf_radius = rdf_radius * 10
print(f"global rdf_radius for downstream workflows: {rdf_radius:.4f} Angstrom")


In [ ]:
import json
import os


def save_radius_dict(radius_dict, filename="rdf_radius.json"):
    with open(os.path.join(os.getcwd(), filename), "w", encoding="utf-8") as f:
        json.dump(radius_dict, f, ensure_ascii=False, indent=2)


save_radius_dict({"rdf_radius": float(rdf_radius)}, filename="rdf_radius.json")


In [ ]:

u = mda.Universe('prod_NVT.tpr','prod_NVT.xtc',indices='index.ndx')


In [ ]:

def get_group_indices(group_name, ndx_filename='index.ndx'):
    
    with open(ndx_filename, 'r') as ndx_file:
        in_group = False
        indices = []
        for line in ndx_file:
            
            if line.startswith('['):
                
                name_in_brackets = line[1:line.find(']')].strip()
                
                in_group = (group_name == name_in_brackets)
                continue
            
            if in_group and line.strip():
                indices.extend(map(int, line.split()))
        return indices


In [ ]:

ion_dict = {}
ion_indices = get_group_indices(center_name)
ion_dict[center_name] = ion_indices

In [ ]:
ion_dict

In [ ]:

index_center_atom = random.choice(ion_dict[center_name]) - 1
print("Public message removed for release.")

In [ ]:
import os
import random
import subprocess

def extract_coordination_environments(ion_dict, center_name, rdf_radius, N_sample=10, structure_file = 'prod_NVT.gro'):
    
    
    
    if center_name not in ion_dict:
        print("Public message removed for release.")
        return
    
    
    center_indices = ion_dict[center_name]
    
    
    if N_sample > len(center_indices):
        print("Public message removed for release.")
        N_sample = len(center_indices)
        
    
    selected_indices = random.sample(center_indices, N_sample)
    
    selected_indices_vmd = [idx - 1 for idx in selected_indices]
    
    
    for idx in selected_indices_vmd:
        print("Public message removed for release.")
    
    
    output_dir = 'coordination_structure_to_compute'
    os.makedirs(output_dir, exist_ok=True)
    
    
    if not os.path.isfile(structure_file):
        print("Public message removed for release.")
        return
    
    
    tcl_script = 
    for idx in selected_indices_vmd:
        mol2_filename = f"coordination_{center_name}_{idx}.mol2"
        tcl_script += 
    
    
    tcl_script += "\nquit\n"
    
    
    tcl_filename = 'extract_coordination.tcl'
    with open(tcl_filename, 'w') as tcl_file:
        tcl_file.write(tcl_script)
    
    
    try:
        
        vmd_executable = 'vmd'
        if not shutil.which(vmd_executable):
            print("Public message removed for release.")
            return
        
        print("Public message removed for release.")
        subprocess.run([vmd_executable, '-dispdev', 'none', '-e', tcl_filename], check=True)
        print("Public message removed for release.")
    except subprocess.CalledProcessError as e:
        print("Public message removed for release.")
    finally:
        
        if os.path.exists(tcl_filename):
            os.remove(tcl_filename)

In [ ]:
extract_coordination_environments(ion_dict, center_name, rdf_radius)

In [ ]:



tcl_coordination = 


In [ ]:






vmd_process = subprocess.Popen(["vmd", "-dispdev", "text"], stdin=subprocess.PIPE, stdout=subprocess.PIPE)
vmd_process.communicate(input=tcl_coordination.encode('utf-8'))

In [ ]:


tcl_coordination_gif = 


In [ ]:

output_dir = "coordination_gif"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)







vmd_process = subprocess.Popen(["vmd", "-dispdev", "text"], stdin=subprocess.PIPE, stdout=subprocess.PIPE)
vmd_process.communicate(input=tcl_coordination_gif.encode('utf-8'))


image_files = [os.path.join(output_dir, f) for f in sorted(os.listdir(output_dir)) if f.endswith('.tif')]
gif_output_file = f'animation_{center_name}_{header_for_min_extrema_str}.gif'
mp4_output_file = f'animation_{center_name}_{header_for_min_extrema_str}.mp4'


images = [imageio.imread(filename) for filename in image_files]

if not images:
    print("No images found. Check the paths and file names.")
else:
    
    imageio.mimsave(gif_output_file, images, duration=200)  



writer = imageio.get_writer(mp4_output_file, fps=5)  
for filename in image_files:
    writer = imageio.get_writer(
    mp4_output_file,
    fps=5,
    format="FFMPEG",
    codec="libx264",      
    pixelformat="yuv420p"
)
writer.close()

In [ ]:



df = pd.read_excel('System.xlsx')


group_indices_dict = {}


for group_name in df['Name']: 
    
    indices = get_group_indices(group_name)

    print(group_name) 
    
    
    if indices:
        
        first_index = indices[0] - 1
        last_index = indices[-1] - 1

        
        group_indices_dict[group_name] = [first_index, last_index]


print(group_indices_dict)


In [ ]:
df

In [ ]:
def tcl_component(component_name, indices):
    index_start, index_end = indices  
    tcl_script = 
    return tcl_script


In [ ]:

for component_name, indices in group_indices_dict.items():
    
    tcl_script = tcl_component(component_name, indices)
    
    
    vmd_process = subprocess.Popen(["vmd", "-dispdev", "text"], stdin=subprocess.PIPE, stdout=subprocess.PIPE)
    vmd_process.communicate(input=tcl_script.encode('utf-8'))

In [ ]:



tcl_snapshot = 


In [ ]:






vmd_process = subprocess.Popen(["vmd", "-dispdev", "text"], stdin=subprocess.PIPE, stdout=subprocess.PIPE)
vmd_process.communicate(input=tcl_snapshot.encode('utf-8'))

In [ ]:

delta_G_df = pd.DataFrame()
delta_G_df['r (nm)'] = df_rdf['r (nm)']


for column in df_rdf.columns[1:]:
    g_r = df_rdf[column]
    delta_G_r_star = k_b * T * np.log(g_r)
    
    delta_G_r_star_kJ_per_mol = delta_G_r_star * (6.02214076e23) / 1000 
    delta_G_df[column] = delta_G_r_star_kJ_per_mol


delta_G_df.to_excel("PMF.xlsx",index=None)


plt.figure(figsize=(3.5, 3), dpi=300)
for column in delta_G_df.columns[1:]:
    plt.plot(delta_G_df['r (nm)'], delta_G_df[column], label=column)

plt.xlabel('r (nm)')
plt.ylabel(r'$\Delta G(r^\ast)$ (kJ/mol)')
plt.xlim(0, 2)  
plt.legend()
plt.tight_layout()  

plt.savefig('PMF.png', format='png', dpi=300, bbox_inches='tight')  
plt.show()  


In [ ]:

def merge_msd_xvg_data(excel_filename):
    
    system_df = pd.read_excel(excel_filename)
    names = system_df['Name'].tolist()

    
    data_frames = []

    for i, name in enumerate(names):
        filename = f"{name}_msd.xvg"
        df = read_xvg(filename)
        
        if i == 0:
            time_df = df.iloc[:, [0]]
        
        df = df.iloc[:, [1]]
        df.columns = [name]  
        data_frames.append(df)

    
    merged_msd_df = pd.concat(data_frames, axis=1)

    
    merged_msd_df.insert(0, 't (ns)', time_df.iloc[:, 0] / 1000)  

    return merged_msd_df

In [ ]:

merged_msd_df = merge_msd_xvg_data("System.xlsx")
print(merged_msd_df)

In [ ]:

plt.figure(figsize=(3.5,3), dpi=300)
for column in merged_msd_df.columns[1:]:
    plt.plot(merged_msd_df['t (ns)'], merged_msd_df[column], label=column)
plt.xlabel('t (ns)')
plt.ylabel(r'MSD (nm$^2$)')
plt.xlim(0, final_time)  
plt.legend()
plt.tight_layout()  
plt.savefig('MSD.png', format='png', dpi=300, bbox_inches='tight')  
plt.show()  

In [ ]:

def gmx_select(name, center_name, rdf_radius, excel_file='System.xlsx', trr_file='prod_NVT.trr', tpr_file='prod_NVT.tpr', ndx_file='index.ndx'):
    
    
    df = pd.read_excel(excel_file)

    
    assert 'Name' in df.columns and 'Index Group Number' in df.columns, "Excel file must contain 'Name' and 'Index Group Number' columns."

    
    try:
        ref_index = df[df['Name'] == center_name]['Index Group Number'].values[0]
    except IndexError:
        raise ValueError(f"Name '{center_name}' not found in the Excel file.")

    
    all_indices = df['Index Group Number'].tolist()

    
    rdf_radius = rdf_radius / 10
    size_file = f'{center_name}_{name}_size.xvg'
    index_file = f'{center_name}_{name}_dynamic_index.ndx'
    mask_file = f'{center_name}_{name}_mask.xvg'
    lifetime_file = f'{center_name}_{name}_lifetime.xvg'
    occupancy_file = f'{center_name}_{name}_occupancy.xvg'

    
    cmd = [
        'gmx', 'select', '-f', trr_file, '-s', tpr_file, '-n', ndx_file,
        '-os', size_file, '-on', index_file, '-om', mask_file,
        '-olt', lifetime_file, '-of', occupancy_file,
    ]
    input_commands = f'group "{name}" and same residue as within {rdf_radius} of group "{center_name}"\n'
    print(input_commands)

    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    stdout, stderr = process.communicate(input=input_commands)
    if process.returncode != 0:
        raise RuntimeError(
            f'gmx select failed for component={name}, center={center_name}, ndx={ndx_file}, returncode={process.returncode}.\n'
            f'Command: {" ".join(cmd)}\n'
            f'Selection: {input_commands}\n'
            f'STDOUT:\n{stdout}\nSTDERR:\n{stderr}'
        )

    def wait_for_files(*filenames, timeout=600):
        
        start_time = time.time()
        while True:
            missing_files = [filename for filename in filenames if not os.path.isfile(filename)]
            if not missing_files:
                return
            if (time.time() - start_time) > timeout:
                raise TimeoutError(
                    f'Timeout waiting for gmx select outputs after {timeout}s. '
                    f'component={name}, center={center_name}, missing={missing_files}, command={" ".join(cmd)}'
                )
            time.sleep(1)

    wait_for_files(size_file, index_file, mask_file, lifetime_file, occupancy_file, timeout=600)
    print(f"Files for {name} around {center_name} are ready for further processing.")
    return size_file, index_file


In [ ]:
center_name

In [ ]:



df = pd.read_excel('System.xlsx')

all_indices = df['Name'].tolist()

with ProcessPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(gmx_select, name, center_name, rdf_radius) for name in all_indices]

    
    for future in as_completed(futures):
        
        try:
            result = future.result()
            
        except Exception as e:
            print(f"Function raised an exception: {e}")

In [ ]:

def get_atom_count_from_mol2(mol2_path):
    
    
    atom_count_regex = re.compile(r'^\s*\d+\s+\d+')
    with open(mol2_path, 'r') as file:
        for line in file:
            
            if atom_count_regex.match(line):
                
                atom_count = int(line.split()[0])
                return atom_count
    return 0  

In [ ]:


def merge_coordination_xvg_data(excel_filename, center_name):
    
    system_df = pd.read_excel(excel_filename)
    names = system_df['Name'].tolist()

    
    data_frames = []
    number_center_name = system_df.loc[system_df['Name'] == center_name, 'Number'].values[0]
    for i, name in enumerate(names):
        filename = f"{center_name}_{name}_size.xvg"
        df = read_xvg(filename)
        
        mol2_path = f"{name}.mol2"
        
        n_name = get_atom_count_from_mol2(mol2_path)
        
        
        if i == 0:
            time_df = df.iloc[:, [0]]
        
        df = (df.iloc[:, [1]] / n_name) / number_center_name
        df.columns = [name]  
        data_frames.append(df)

    
    merged_coordination_df = pd.concat(data_frames, axis=1)

    
    merged_coordination_df.insert(0, 't (ns)', time_df.iloc[:, 0] / 1000)  

    return merged_coordination_df

In [ ]:
merged_coordination_df = merge_coordination_xvg_data('System.xlsx', center_name)

In [ ]:

plt.figure(figsize=(3.5,3), dpi=300)
for column in merged_coordination_df.columns[1:]:
    plt.plot(merged_coordination_df['t (ns)'], merged_coordination_df[column], label=column)
plt.xlabel('t (ns)')
plt.ylabel(r'Coordination Number')
plt.xlim(0, final_time)  
plt.legend()
plt.tight_layout()  

plt.savefig(f'{center_name}_selection_size.png', format='png', dpi=300, bbox_inches='tight')  
plt.show()  

In [ ]:
import pandas as pd


def fill_average_coordination_numbers(merged_coordination_df: pd.DataFrame, 
                                      df_system: pd.DataFrame) -> pd.DataFrame:
    
    
    
    def get_mean_coordination(name):
        
        
        if name in merged_coordination_df.columns:
            return merged_coordination_df[name].mean()
        else:
            return None  

    df_system['average_coordination_number'] = df_system['Name'].apply(get_mean_coordination)
    
    return df_system

In [ ]:

df = fill_average_coordination_numbers(merged_coordination_df, df)
df.to_excel("System.xlsx",index=None)

In [ ]:
import os
import re
import subprocess
import pandas as pd

def calculate_molecular_volume(df_system: pd.DataFrame) -> pd.DataFrame:
    
    
    
    df_system["Mol_Volume(Angstrom^3)"] = None

    
    
    commands = "12\n6\n-1\n-1\n-1\nq\n"

    
    
    
    pattern = re.compile(r"Volume:\s*[\d.]+\s*Bohr\^3\s*\(\s*([\d.]+)\s*Angstrom\^3\)")

    
    for index, row in df_system.iterrows():
        
        name = row["Name"]

        
        mol2_path = os.path.join(os.getcwd(), f"{name}.mol2")

        
        if os.path.isfile(mol2_path):
            
            process = subprocess.Popen(
                ["Multiwfn", mol2_path],
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )

            
            stdout, stderr = process.communicate(commands)

            
            match = pattern.search(stdout)
            if match:
                
                volume_value = match.group(1)

                
                df_system.at[index, "Mol_Volume(Angstrom^3)"] = float(volume_value)
        else:
            
            pass
    
    
    if "average_coordination_number" in df_system.columns and "Mol_Volume(Angstrom^3)" in df_system.columns:
        
        df_system["Solvation_Shell_Volume(Angstrom^3)"] = (
            df_system["average_coordination_number"] * df_system["Mol_Volume(Angstrom^3)"]
        )

    return df_system

In [ ]:

df = pd.read_excel("System.xlsx")
df = calculate_molecular_volume(df)
df.to_excel("System.xlsx",index=None)
df

In [ ]:
import re

def compute_average_residence_time(input_file, output_trs=None, output_frq=None):
    

    
    
    
    
    if output_trs is None:
        
        output_trs = f"{input_file.rsplit('.', 1)[0]}_trs.dat"
    if output_frq is None:
        
        output_frq = f"{input_file.rsplit('.', 1)[0]}_frq.dat"

    
    
    
    
    with open(input_file, 'r') as f:
        lines = f.readlines()

    
    valid_lines = []
    for line in lines:
        line_strip = line.strip()
        if line_strip and not line_strip.startswith('#') and not line_strip.startswith('@'):
            valid_lines.append(line_strip)

    
    if not valid_lines:
        print("Public message removed for release.")
        return

    
    
    
    
    first_valid_line_split = valid_lines[0].split()
    Ncol = len(first_valid_line_split)

    
    
    
    
    
    
    
    
    
    trs_lines = []
    
    for col_idx in range(1, Ncol):
        col_data_str = ""
        for line in valid_lines:
            
            split_line = line.split()
            if len(split_line) == Ncol:
                
                col_data_str += split_line[col_idx]
        
        trs_lines.append(col_data_str)

    
    with open(output_trs, 'w') as f_trs:
        for line_str in trs_lines:
            f_trs.write(line_str + "\n")

    
    
    
    
    
    
    if len(valid_lines) < 2:
        print("Public message removed for release.")
        dt = 0.0
    else:
        try:
            time1 = float(valid_lines[0].split()[0])
            time2 = float(valid_lines[1].split()[0])
            dt = time2 - time1
        except ValueError:
            print("Public message removed for release.")
            dt = 0.0

    
    
    
    
    
    
    
    
    
    Navg = 0
    Tavg = 0.0

    with open(output_frq, 'w') as f_frq:
        for line_str in trs_lines:
            
            replaced_str = re.sub(r'0+', ' ', line_str)

            
            segments = replaced_str.split()
            for seg in segments:
                T = len(seg)
                f_frq.write(f"{T}\n")
                Navg += 1
                Tavg += T

        
        if Navg > 0:
            avg_res_time = Tavg * dt / Navg
        else:
            avg_res_time = 0.0

        
        f_frq.write(f"# Avaraged Residence Time(ps)= {avg_res_time}\n")

    
    
    
    print("Public message removed for release.")
    print("Public message removed for release.")
    return avg_res_time

In [ ]:

import os
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed

def _worker(task):
    
    index, name, center_name = task

    
    mask_path = f"{center_name}_{name}_mask.xvg"

    if not os.path.exists(mask_path):
        
        print("Public message removed for release.")
        return index, None
    else:
        
        avg_res_time = compute_average_residence_time(mask_path)
        return index, avg_res_time

def add_average_residence_time_parallel(df_system, max_workers=5):
    
    
    tasks = []
    for index, row in df_system.iterrows():
        name = row["Name"]
        center_name = row["center atom"]
        tasks.append((index, name, center_name))

    
    results_dict = {}

    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        
        future_to_task = {executor.submit(_worker, task): task for task in tasks}

        
        for future in as_completed(future_to_task):
            task = future_to_task[future]
            try:
                
                index, avg_res_time = future.result()
                results_dict[index] = avg_res_time
            except Exception as exc:
                
                print("Public message removed for release.")
                results_dict[task[0]] = None

    
    for index, avg_res_time in results_dict.items():
        df_system.loc[index, "avg_res_time(ps)"] = avg_res_time

    return df_system

In [ ]:

import os
import re
import pandas as pd

def parse_residence_time(df_system):
    
    
    
    if 'avg_res_time(ps)' not in df_system.columns:
        df_system['avg_res_time(ps)'] = float('nan')
    
    
    
    
    pattern = re.compile(r'^#\s*Avaraged Residence Time\(ps\)=\s*(\S+)')
    
    
    for index, row in df_system.iterrows():
        name = row["Name"]
        center_name = row["center atom"]
        
        
        filename = f"{center_name}_{name}_mask_frq.dat"
        print(filename)
        
        
        if os.path.isfile(filename):
            with open(filename, 'r', encoding='utf-8') as f:
                for line in f:
                    
                    match = pattern.match(line.strip())
                    if match:
                        
                        avg_res_time = float(match.group(1))
                        
                        df_system.at[index, 'avg_res_time(ps)'] = avg_res_time
                        break  
            
        else:
            
            
            pass
    
    return df_system

In [ ]:

df = pd.read_excel("System.xlsx")
df = add_average_residence_time_parallel(df, max_workers=5) 
df = parse_residence_time(df) 
df.to_excel("System.xlsx",index=None)
df

In [ ]:

def merge_lifetime_xvg_data(excel_filename, center_name):
    
    system_df = pd.read_excel(excel_filename)
    names = system_df['Name'].tolist()

    
    data_frames = []
    number_center_name = system_df.loc[system_df['Name'] == center_name, 'Number'].values[0]
    for i, name in enumerate(names):
        filename = f"{center_name}_{name}_lifetime.xvg"
        df = read_xvg(filename)
        
        mol2_path = f"{name}.mol2"
        
        n_name = get_atom_count_from_mol2(mol2_path)
        
        
        if i == 0:
            time_df = df.iloc[:, [0]]
        
        df = (df.iloc[:, [1]] / n_name) / number_center_name
        df.columns = [name]  
        data_frames.append(df)

    
    merged_lifetime_df = pd.concat(data_frames, axis=1)

    
    merged_lifetime_df.insert(0, 't (ns)', time_df.iloc[:, 0] / 1000)  

    return merged_lifetime_df

In [ ]:
merged_lifetime_df = merge_lifetime_xvg_data('System.xlsx', center_name)

In [ ]:

plt.figure(figsize=(3.5,3), dpi=300)
for column in merged_lifetime_df.columns[1:]:
    plt.plot(merged_lifetime_df['t (ns)'], merged_lifetime_df[column], label=column)
plt.xlabel('t (ns)')
plt.ylabel(r'Number of occurrences')
plt.xlim(0, final_time)  
plt.legend()
plt.tight_layout()  
plt.savefig(f'{center_name}_lifetime.png', format='png', dpi=300, bbox_inches='tight')  
plt.show()  

In [ ]:

df = pd.read_excel('System.xlsx')
df

In [ ]:

name_to_index = pd.Series(df['Index Group Number'].values, index=df['Name']).to_dict()

In [ ]:
name_to_index

In [ ]:

def run_gmx_densmap(name, index_number, aver='z',trr_file='prod_NVT.trr', tpr_file='prod_NVT.tpr', ndx_file='index.ndx'):
    cmd = ['gmx', 'densmap', '-f', trr_file, '-s', tpr_file, '-n', ndx_file, '-aver', aver, '-o', f'{name}_densmap_{aver}.xpm']
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    
    process.stdin.write(f"{index_number}\n")
    process.stdin.flush()

    
    stdout, stderr = process.communicate()

    if stderr:
        print(f"calculating densmap for {name}:")
        print(stderr)

In [ ]:

with ProcessPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(run_gmx_densmap, name, index_number) for name, index_number in name_to_index.items()]

    
    for future in as_completed(futures):
        
        try:
            result = future.result()
            
        except Exception as e:
            print(f"Function raised an exception: {e}")

In [ ]:

densmap_xpm_list = [xpm for xpm in os.listdir('.') if xpm.endswith('.xpm')]

print(densmap_xpm_list)

In [ ]:

def run_gmx_xpm2ps(xpm_file):
    base_xpm_filename = os.path.splitext(xpm_file)[0]
    cmd = ['gmx', 'xpm2ps', '-f', xpm_file, '-o', f'{base_xpm_filename}.eps']
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    
    stdout, stderr = process.communicate()

    if stderr:
        print(f"calculating densmap for {xpm_file}:")
        print(stderr)

In [ ]:

with ProcessPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(run_gmx_xpm2ps, xpm_file) for xpm_file in densmap_xpm_list]

    
    for future in as_completed(futures):
        
        try:
            result = future.result()
            
        except Exception as e:
            print(f"Function raised an exception: {e}")

In [ ]:

u_prod = mda.Universe('prod_NVT.tpr','prod_NVT.xtc',indices='index.ndx')



prod_frames = len(u_prod.trajectory)


prod_volume = u_prod.dimensions[0] * u_prod.dimensions[1] * u_prod.dimensions[2] * 1e-30

In [ ]:

start_frame = max(prod_frames - 1000, 0)


with mda.Writer("prod_NVT_last_1000_frames.xtc", u_prod.atoms.n_atoms) as W:
    
    for ts in u_prod.trajectory[start_frame:]:
        W.write(u_prod)

In [ ]:

u_analysis = mda.Universe('prod_NVT.tpr','prod_NVT_last_1000_frames.xtc',indices='index.ndx')

analysis_frames = len(u_analysis.trajectory)
print(analysis_frames)

In [ ]:

def compute_average_structure_factor(u, rdf_radius, n_q_bins=100):
    
    
    
    box_size = u.dimensions[:3]
    
    
    N = u.atoms.n_atoms
    
    
    Sq = np.zeros(n_q_bins)
    q_values = np.linspace(0, np.pi / rdf_radius, n_q_bins)
    
    
    for ts in u.trajectory:
        
        positions = u.atoms.positions
        
        
        for i, q in enumerate(q_values[1:], start=1):  
            
            for x in range(-1, 2, 2):  
                for y in range(-1, 2, 2):
                    for z in range(-1, 2, 2):
                        q_vec = q * np.array([x, y, z]) / np.sqrt(x**2 + y**2 + z**2)
                        rho_q = np.sum(np.exp(-1j * np.dot(positions, q_vec)))
                        Sq[i] += np.abs(rho_q)**2
    
    
    N_frames = len(u.trajectory)
    Sq /= (N * N_frames * 8)  
    
    
    return q_values[1:], Sq[1:]

In [ ]:

q_values, Sq = compute_average_structure_factor(u_analysis, rdf_radius/10)


with open('structure_factor.txt', 'w') as f:
    f.write('q (nm⁻¹)\tS(q)\n')  
    for q, s in zip(q_values, Sq):
        f.write(f'{q}\t{s}\n')  

In [ ]:

plt.figure(figsize=(3.5,3), dpi=300)
plt.plot(q_values, Sq, marker='o', linestyle='-', color='blue')
plt.xlabel('Wave vector magnitude q (nm⁻¹)')
plt.ylabel('S(q)')
plt.title('Structure Factor S(q)')
plt.ylim(0, 3)
plt.grid(True)

plt.tight_layout()  

plt.savefig('structure_factor.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:

def run_gmx_dipoles(trr_file='prod_NVT.trr', tpr_file='prod_NVT.tpr', delete_file=True):
    cmd = ['gmx', 'dipoles', '-f', trr_file, '-s', tpr_file, '-corr', 'total', '-eps', 'epsilon.xvg', '-c', 'dipcorr.xvg', '-g', 'KirkwoodGfactor.xvg']
    output_file = 'gmx_dipoles_output.txt'
    
    with open(output_file, 'w') as file:
        process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=file, stderr=file, text=True)
        
        
        output, error = process.communicate(input='0\n')

    
    process.wait()
    
    
    with open(output_file, 'r') as file:
        output = file.read()
        match = re.search(r'\bEpsilon\s*=\s*(\d+\.\d+)', output)
        if match:
            
            epsilon_value_dipoles = match.group(1)
            print(f"Static dielectric constant: epsilon={epsilon_value_dipoles}")
            return epsilon_value_dipoles
        else:
            print("Could not find the static dielectric constant in the output.")
            return None
    
    
    if delete_file:
        os.remove(output_file)
    
    print("The 'Absolute values:' line could not be found in the output.")
    return None

In [ ]:
epsilon_value_dipoles = run_gmx_dipoles(trr_file='prod_NVT.trr', tpr_file='prod_NVT.tpr', delete_file=True)

In [ ]:

def run_gmx_dielectric(epsilon_value_dipoles, trr_file='prod_NVT.trr', tpr_file='prod_NVT.tpr'):
    cmd = ['gmx', 'dielectric', '-f', 'dipcorr.xvg', '-eps0', epsilon_value_dipoles, '-ffn', 'aexp', '-o', 'epsw.xvg']
    
    process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    output, error = process.communicate()

    
    process.wait()

In [ ]:
run_gmx_dielectric(epsilon_value_dipoles)

In [ ]:
df_epsw = read_xvg('epsw.xvg')

In [ ]:

plt.figure(figsize=(3.5,3), dpi=300)
plt.plot(df_epsw[0], df_epsw[1])
plt.xlabel('Frequency (GHz)')
plt.ylabel('eps')
plt.legend()
plt.tight_layout()  
plt.savefig(f'dielectric_constant_spectrum.png', format='png', dpi=300, bbox_inches='tight')  
plt.show()  

def plot_strain_stress(df, strain_col="Strain", stress_col="Stress (MPa)", name=None, output=False, figsize=(3.5,3)):
    # Create a copy of the DataFrame to avoid modifying the original data
    df_copy = df.copy()
    
    # Apply median filter to the converted 'Stress-Pxx(bar)' column
    df_copy['Smoothed Stress'] = medfilt(df_copy[stress_col], kernel_size=31)  # kernel size can be adjusted
    
    # Plotting both original and smoothed data
    plt.figure(figsize=figsize, dpi=300)
    plt.plot(df_copy[strain_col], df_copy[stress_col], label='Original Stress')
    plt.plot(df_copy[strain_col], df_copy['Smoothed Stress'], label='Smoothed Stress', linestyle='--')
    plt.xlabel('Strain')
    plt.ylabel('Stress (MPa)')
    plt.legend()
    plt.grid(True)
    if output:
        plt.savefig(f"{name}_strain_stress_plot.tif", format='tif', dpi=300)

    plt.show()

if polymer_name:
    plot_strain_stress(df_strain_stress, name="System", output=True, figsize=(3.5,3))

In [ ]:
def extract_polymer_index(df):
    

    
    required_columns = {'is polymer', 'Name', 'Index Group Number'}
    if not required_columns.issubset(df.columns):
        missing_columns = required_columns - set(df.columns)
        raise KeyError("Public message removed for release.")

    
    polymer_name_to_index = {}

    
    for _, row in df.iterrows():
        
        if row['is polymer']:
            
            name = row['Name']
            index_number = row['Index Group Number']
            
            
            if not pd.api.types.is_number(index_number):
                raise ValueError("Public message removed for release.")

            
            polymer_name_to_index[name] = index_number

    
    return polymer_name_to_index

if polymer_name:
    df_merged_gyrate = plot_after_stretch_polymer_gyrate(output=True)

if polymer_name:
    df_merged_endtoend_distance = plot_after_stretch_polymer_endtoend_distance(output=True)